# Import

In [1]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv

import kagglehub
import hiddenlayer as hl
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import torch.nn.functional as F

load_dotenv()
DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f'Using {DEVICE} for inference')

/home/redduck/VSProjects/ITMO_SECS_CV_2025/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using cpu for inference


In [2]:
from utils import *
from models import CustomResNet50, SimpleCNN

# Data

In [3]:
data_dir = str('data/input/PetImages')
train_loader, val_loader, test_loader = get_dataloaders(data_dir)

Downlod dataset to path: /home/redduck/.cache/kagglehub/datasets/bhavikjikadara/dog-and-cat-classification-dataset/versions/1/PetImages
An unexpected error occurred: Destination path '/home/redduck/VSProjects/ITMO_SECS_CV_2025/lab3/data/input/PetImages' already exists


# Model

## Vis arch

In [4]:
# # https://github.com/waleedka/hiddenlayer/blob/master/demos/pytorch_graph.ipynb
# transforms = [hl.transforms.Prune('Constant')] # Removes Constant nodes from graph.

# # Rather than using the default transforms, build custom ones to group
# # nodes of residual and bottleneck blocks.
# transforms = [
#     # Fold Conv, BN, RELU layers into one
#     hl.transforms.Fold("Conv > BatchNorm > Relu", "ConvBnRelu"),
#     # Fold Conv, BN layers together
#     hl.transforms.Fold("Conv > BatchNorm", "ConvBn"),
#     # Fold bottleneck blocks
#     hl.transforms.Fold("""
#         ((ConvBnRelu > ConvBnRelu > ConvBn) | ConvBn) > Add > Relu
#         """, "BottleneckBlock", "Bottleneck Block"),
#     # Fold residual blocks
#     hl.transforms.Fold("""ConvBnRelu > ConvBnRelu > ConvBn > Add > Relu""",
#                        "ResBlock", "Residual Block"),
#     # Fold repeated blocks
#     hl.transforms.FoldDuplicates(),
# ]

# # Display graph using the transforms above
# graph = hl.build_graph(resnet50, torch.zeros([1, 3, 224, 224]), transforms=transforms)
# graph.theme = hl.graph.THEMES['blue'].copy()
# graph.save('rnn_hiddenlayer', format='png')

## Train

In [5]:
model = SimpleCNN().to(DEVICE)

# Parameters
epochs = 5
lr = 1e-3

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

In [9]:
train_losses, val_losses, train_accs, val_accs = train_model(
    model,
    train_loader, val_loader,
    epochs,
    criterion,
    optimizer,
    DEVICE
)

  6%|▌         | 36/625 [00:34<11:15,  1.15s/it]/home/redduck/VSProjects/ITMO_SECS_CV_2025/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
100%|██████████| 625/625 [08:56<00:00,  1.16it/s]


Epoch [1/5] Train Loss: 0.6585 | Train Acc: 0.5959 Val Loss: 0.6548 | Val Acc: 0.5931


 51%|█████     | 320/625 [04:24<04:02,  1.26it/s]/home/redduck/VSProjects/ITMO_SECS_CV_2025/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
100%|██████████| 625/625 [08:41<00:00,  1.20it/s]


Epoch [2/5] Train Loss: 0.6402 | Train Acc: 0.6161 Val Loss: 0.6318 | Val Acc: 0.6261


 53%|█████▎    | 332/625 [04:31<03:43,  1.31it/s]/home/redduck/VSProjects/ITMO_SECS_CV_2025/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
100%|██████████| 625/625 [08:40<00:00,  1.20it/s]


...

 91%|█████████ | 570/625 [08:27<00:51,  1.08it/s]/home/redduck/VSProjects/ITMO_SECS_CV_2025/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
100%|██████████| 625/625 [09:18<00:00,  1.12it/s]


Epoch [4/5] Train Loss: 0.5978 | Train Acc: 0.6635 Val Loss: 0.6006 | Val Acc: 0.6631


 15%|█▌        | 96/625 [01:10<06:13,  1.42it/s]/home/redduck/VSProjects/ITMO_SECS_CV_2025/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
100%|██████████| 625/625 [08:42<00:00,  1.20it/s]


Epoch [5/5] Train Loss: 0.5773 | Train Acc: 0.6927 Val Loss: 0.5892 | Val Acc: 0.6695


SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (gap): AdaptiveAvgPool2d(output_size=1)
  (fc1): Linear(in_features=32, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
)

## Evaluate

In [10]:
evaluate(model, test_loader, criterion, DEVICE)

(0.7210007954616936, 0.5306122448979592)

In [8]:
save_path = 'output/graph_vis.png'
visualize_losses_metrics(
    train_losses,
    val_losses,
    train_accs,
    val_accs,
    save_path,
    figsize=(10, 4),
    dpi=300,
    show=False
)

# ResNet

In [ ]:
model = CustomResNet50().to(DEVICE)

In [ ]:
epochs = 5
lr = 1e-3

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)